# 07 — Final comparison

Headline figures for the report. Reads `results/depth_sweep.json`,
`results/size_scaling.json`, `results/risk_sweep.json` — no heavy
computation, just plotting. Run notebooks 04, 05, 06 first.

In [ ]:
# === Bootstrap (Colab + local) ===
import sys, os, json
try:
    import google.colab  # noqa: F401
    get_ipython().system('test -d /content/fys5419 || git clone -q https://github.com/egil10/fys5419.git /content/fys5419')
    get_ipython().run_line_magic('cd', '/content/fys5419/project2/code/notebooks')
except ImportError:
    pass
sys.path.append('..')
from scripts.colab import setup; setup()

# === Project imports ===
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.plotting import apply_style, PALETTE, title, fig_path
apply_style()

RESULTS = Path.cwd().parent / 'results'

### Figure 1 — Depth sweep (Sweep 1)

In [ ]:
sweep = pd.DataFrame(json.loads((RESULTS / 'depth_sweep.json').read_text()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(sweep.p, sweep.ratio, 'o-', color=PALETTE['blue'], lw=2, ms=8)
axes[0].axhline(1.0, color=PALETTE['red'], ls='--', lw=1.2, label='optimum')
axes[0].set_xticks(sweep.p); axes[0].set_xlabel('depth p')
axes[0].set_ylabel('approximation ratio E / E_opt')
title(axes[0], 'QAOA approximation ratio', 'closer to 1 is better')
axes[0].legend()

axes[1].plot(sweep.p, sweep.p_optimal,  'o-', color=PALETTE['red'],         lw=2, ms=8, label='P(optimum)')
axes[1].plot(sweep.p, sweep.p_feasible, 'o-', color=PALETTE['blue_muted'], lw=2, ms=8, label='P(feasible)')
axes[1].set_xticks(sweep.p); axes[1].set_xlabel('depth p'); axes[1].set_ylabel('probability')
title(axes[1], 'Measurement probabilities', 'higher = more concentrated')
axes[1].legend()

plt.tight_layout()
fig.savefig(fig_path('compare', 'depth_sweep'), bbox_inches='tight')
plt.show()

### Figure 2 — Size scaling (Sweep 2)

In [ ]:
# Size-scaling sweep is now produced as one record per (n, solver, subset).
# Aggregate to median + IQR across subsets for each (n, solver).
scale = pd.DataFrame(json.loads((RESULTS / 'size_scaling.json').read_text()))
agg = (scale
       .groupby(['n', 'solver'])
       .agg(median_ratio=('ratio', 'median'),
            q25_ratio   =('ratio', lambda s: s.quantile(0.25)),
            q75_ratio   =('ratio', lambda s: s.quantile(0.75)),
            median_rt   =('runtime_s', 'median'),
            n_subsets   =('ratio', 'count'))
       .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for solver, sub in agg.groupby('solver'):
    sub = sub.sort_values('n')
    line, = axes[0].plot(sub.n, sub.median_ratio, 'o-', lw=2, ms=7, label=solver)
    axes[0].fill_between(sub.n, sub.q25_ratio, sub.q75_ratio,
                         color=line.get_color(), alpha=0.18)
    axes[1].semilogy(sub.n, sub.median_rt, 'o-', lw=2, ms=7,
                     color=line.get_color(), label=solver)

axes[0].axhline(1.0, color=PALETTE['charcoal'], ls=':', lw=1)
axes[0].set_xlabel('problem size n'); axes[0].set_ylabel('cost / brute-force cost')
title(axes[0], 'Quality vs problem size',
      'median +/- IQR over random size-n subsets; 1.0 = brute-force optimum')
axes[0].legend(fontsize=8)

axes[1].set_xlabel('problem size n'); axes[1].set_ylabel('runtime (s, log)')
title(axes[1], 'Runtime vs problem size', 'median across subsets')
axes[1].legend(fontsize=8)

plt.tight_layout()
fig.savefig(fig_path('compare', 'scaling'), bbox_inches='tight')
plt.show()

### Figure 3 — Risk landscape (Sweep 3)

In [ ]:
risk = pd.DataFrame(json.loads((RESULTS / 'risk_sweep.json').read_text()))

fig, ax = plt.subplots(figsize=(10, 5))
for solver, sub in risk.groupby('solver'):
    ax.semilogx(sub['lambda'], sub.ratio, 'o-', lw=2, ms=7, label=solver)
ax.axhline(1.0, color=PALETTE['charcoal'], ls=':', lw=1)
ax.set_xlabel('risk aversion lambda (log)'); ax.set_ylabel('cost / brute-force cost')
title(ax, 'Approximation quality vs risk aversion',
      'low lambda: near-linear (greedy wins); high lambda: frustrated (QAOA mechanism matters)')
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(fig_path('compare', 'risk'), bbox_inches='tight')
plt.show()